# Candle Anatomy — Classification, Continuation & Retracement

For every candle on every timeframe:
1. Classify its shape (wick/body ratios + body location) into one of your reference-chart types
2. Bucket where it **opened** relative to the previous same-timeframe candle's range
3. For three forward-looking horizons — rest of session, fixed N bars, and a bounded window sized
   for "time to 2x the candle's own range" — measure:
   - **Continuation**: upside & downside max-favorable-move, as % of close
   - **Retracement**: whether/how deep price comes back to the PRIOR candle's open, midpoint, and
     full high/low range (tracked separately)

Same session/day conventions as your other notebooks: NY trading day = 18:00 → 17:00 ET;
sessions Globex 18-20, Asia 20-02, London 02-08, NY AM 08-12, NY PM 12-17.

In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

# ── CONFIG ──────────────────────────────────────────────────────────────────
FILE_PATH  = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
START_DATE = '2013-01-01'
END_DATE   = '2025-12-31'
OUT_DIR    = r"C:\Users\tammm\NQ and ES analysis\candle_anatomy"

TIMEFRAMES = ['5min', '15min', '30min', '1h']

# candle shape thresholds (edit to taste)
DOJI_BODY_MAX         = 0.05
DOJI_WICK_DOMINANT    = 0.90
MARUBOZU_BODY_MIN     = 0.95
SHAVEN_BODY_MIN       = 0.50
SHAVEN_WICK_MAX       = 0.05
BIG_BODY_MIN          = 0.60
HAMMER_BODY_MAX       = 0.30
HAMMER_DOM_WICK_MIN   = 0.60
HAMMER_OTHER_WICK_MAX = 0.10
SPIN_BODY_MIN         = 0.10
SPIN_BODY_MAX         = 0.40
SPIN_WICK_MIN_EACH    = 0.15

# open-position buckets (candle open vs previous candle's range)
SEG_EDGES  = [-100, 0, 33.33, 66.67, 100, 200]
SEG_LABELS = ['Below prior low', 'Lower third', 'Middle third', 'Upper third', 'Above prior high']

# forward-looking horizons
FIXED_N_BARS     = [5, 20]      # "fixed N candles forward"
TARGET_MULTIPLE  = 2.0          # "until price hits Nx the candle's own range"
MAX_BARS_SEARCH  = 150          # cap for the target-touch search (perf/memory bound)

MIN_SAMPLE_SIZE  = 20           # hide summary rows with fewer than this many candles
SAVE_EVENT_LEVEL = True         # per-candle CSV is large; set False to skip saving it

# classic bull/bear/neutral bias per type, used only for the "aligned continuation" heatmap below
CANDLE_BIAS = {
    'Marubozu Bull': 'bull', 'Big White Candle': 'bull', 'Shaven Head Bull': 'bull',
    'Shaven Bottom Bull': 'bull', 'Hammer': 'bull', 'Inverted Hammer': 'bull', 'Dragonfly Doji': 'bull',
    'Marubozu Bear': 'bear', 'Big Black Candle': 'bear', 'Shaven Head Bear': 'bear',
    'Shaven Bottom Bear': 'bear', 'Hanging Man': 'bear', 'Inverted Black Hammer': 'bear', 'Gravestone Doji': 'bear',
    'Doji': 'neutral', 'Spinning Top': 'neutral', 'Unclassified': 'neutral',
}

def hour_to_session(h):
    if 18 <= h < 20: return 'Globex (18-20)'
    if h >= 20 or h < 2: return 'Asia (20-02)'
    if 2 <= h < 8: return 'London (02-08)'
    if 8 <= h < 12: return 'NY AM (08-12)'
    if 12 <= h < 17: return 'NY PM (12-17)'
    return 'Other'

SESSION_ORDER = ['Globex (18-20)', 'Asia (20-02)', 'London (02-08)', 'NY AM (08-12)', 'NY PM (12-17)']

## Step 1 — Candle classification

Vectorized: computes body%/wick% for every row, then assigns a type using the same priority order as the reference chart (dojis first, then marubozu, shaven, big candles, hammer family, spinning top).

In [ ]:
def classify_candles(tf):
    o, h, l, c = tf['Open'].values, tf['High'].values, tf['Low'].values, tf['Close'].values
    rng = h - l
    rng_safe = np.where(rng <= 0, np.nan, rng)
    body = np.abs(c - o)
    body_pct = body / rng_safe
    upper = h - np.maximum(o, c)
    lower = np.minimum(o, c) - l
    up_pct = upper / rng_safe
    lo_pct = lower / rng_safe
    bull = c > o

    tf['body_pct'] = body_pct
    tf['upper_wick_pct'] = up_pct
    tf['lower_wick_pct'] = lo_pct
    tf['direction'] = np.select([bull, c < o], ['bull', 'bear'], default='flat')

    conds = [
        (body_pct <= DOJI_BODY_MAX) & (lo_pct >= DOJI_WICK_DOMINANT),
        (body_pct <= DOJI_BODY_MAX) & (up_pct >= DOJI_WICK_DOMINANT),
        (body_pct <= DOJI_BODY_MAX),
        (body_pct >= MARUBOZU_BODY_MIN) & bull,
        (body_pct >= MARUBOZU_BODY_MIN) & ~bull,
        (up_pct <= SHAVEN_WICK_MAX) & (lo_pct > SHAVEN_WICK_MAX) & (body_pct >= SHAVEN_BODY_MIN) & bull,
        (up_pct <= SHAVEN_WICK_MAX) & (lo_pct > SHAVEN_WICK_MAX) & (body_pct >= SHAVEN_BODY_MIN) & ~bull,
        (lo_pct <= SHAVEN_WICK_MAX) & (up_pct > SHAVEN_WICK_MAX) & (body_pct >= SHAVEN_BODY_MIN) & bull,
        (lo_pct <= SHAVEN_WICK_MAX) & (up_pct > SHAVEN_WICK_MAX) & (body_pct >= SHAVEN_BODY_MIN) & ~bull,
        (body_pct >= BIG_BODY_MIN) & bull,
        (body_pct >= BIG_BODY_MIN) & ~bull,
        (body_pct <= HAMMER_BODY_MAX) & (lo_pct >= HAMMER_DOM_WICK_MIN) & (up_pct <= HAMMER_OTHER_WICK_MAX) & bull,
        (body_pct <= HAMMER_BODY_MAX) & (lo_pct >= HAMMER_DOM_WICK_MIN) & (up_pct <= HAMMER_OTHER_WICK_MAX) & ~bull,
        (body_pct <= HAMMER_BODY_MAX) & (up_pct >= HAMMER_DOM_WICK_MIN) & (lo_pct <= HAMMER_OTHER_WICK_MAX) & bull,
        (body_pct <= HAMMER_BODY_MAX) & (up_pct >= HAMMER_DOM_WICK_MIN) & (lo_pct <= HAMMER_OTHER_WICK_MAX) & ~bull,
        (body_pct >= SPIN_BODY_MIN) & (body_pct <= SPIN_BODY_MAX) & (up_pct >= SPIN_WICK_MIN_EACH) & (lo_pct >= SPIN_WICK_MIN_EACH),
    ]
    choices = [
        'Dragonfly Doji', 'Gravestone Doji', 'Doji',
        'Marubozu Bull', 'Marubozu Bear',
        'Shaven Head Bull', 'Shaven Head Bear',
        'Shaven Bottom Bull', 'Shaven Bottom Bear',
        'Big White Candle', 'Big Black Candle',
        'Hammer', 'Hanging Man',
        'Inverted Hammer', 'Inverted Black Hammer',
        'Spinning Top',
    ]
    tf['candle_type'] = np.select(conds, choices, default='Unclassified')
    return tf

## Step 2 — Build each timeframe & bucket the open vs. the previous candle

In [ ]:
def build_timeframe(df1, rule):
    tf = df1[['Open', 'High', 'Low', 'Close']].resample(rule, label='left', closed='left').agg(
        {'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'}
    ).dropna()
    tf['Hour_ET'] = tf.index.hour
    tf['Session'] = tf['Hour_ET'].map(hour_to_session)
    tf['NY_Trading_Day'] = (tf.index - pd.Timedelta(hours=7)).normalize()
    return tf

def add_open_position(tf):
    prev_high = tf['High'].shift(1)
    prev_low = tf['Low'].shift(1)
    prev_range = prev_high - prev_low
    open_pct = (tf['Open'] - prev_low) / prev_range.replace(0, np.nan) * 100
    tf['prior_open_pct'] = open_pct
    tf['open_position'] = pd.cut(open_pct, bins=SEG_EDGES, labels=SEG_LABELS, include_lowest=True)
    tf['prior_open'] = tf['Open'].shift(1)
    tf['prior_mid'] = (prev_high + prev_low) / 2
    tf['prior_high'] = prev_high
    tf['prior_low'] = prev_low
    return tf

## Step 3 — Forward-looking horizons

Three ways to define "how far forward do we look": rest of the current session, a fixed number of bars, and a bounded window sized for the "until price travels 2x the candle's own range" question (that question's real answer — reach-rate & speed — is computed separately, see `add_target_touch_stats`).

In [ ]:
def forward_group_extreme(values, grp, kind):
    """Max/min of `values` over all FUTURE rows within the same group (excl current row)."""
    nxt = values.groupby(grp, sort=False).shift(-1)
    if kind == 'max':
        return nxt.groupby(grp, sort=False).transform(lambda x: x[::-1].cummax()[::-1])
    return nxt.groupby(grp, sort=False).transform(lambda x: x[::-1].cummin()[::-1])

def fixed_n_extreme(values, n, kind):
    """Max/min of `values` over the next n rows (excl current row)."""
    nxt = values.shift(-1)
    if kind == 'max':
        return nxt[::-1].rolling(window=n, min_periods=1).max()[::-1]
    return nxt[::-1].rolling(window=n, min_periods=1).min()[::-1]

def first_touch_bars(values, target, max_bars, direction):
    """Bars-ahead (1-indexed) until `values` first reaches `target`; -1 if not within max_bars."""
    n = len(values)
    pad_val = np.inf if direction == 'up' else -np.inf
    shifted = np.concatenate([values[1:], np.full(max_bars, pad_val)])
    windows = sliding_window_view(shifted, max_bars)[:n]
    mask = (windows >= target[:, None]) if direction == 'up' else (windows <= target[:, None])
    reached = mask.any(axis=1)
    first_idx = np.argmax(mask, axis=1)
    return np.where(reached, first_idx + 1, -1)

def add_horizon_extremes(tf):
    horizons = {}
    sess_grp = tf['NY_Trading_Day'].astype(str) + '|' + tf['Session'].astype(str)
    horizons['rest_of_session'] = (
        forward_group_extreme(tf['High'], sess_grp, 'max'),
        forward_group_extreme(tf['Low'], sess_grp, 'min'),
    )
    for n in FIXED_N_BARS:
        horizons[f'fixed_{n}_bars'] = (
            fixed_n_extreme(tf['High'], n, 'max'),
            fixed_n_extreme(tf['Low'], n, 'min'),
        )
    horizons[f'bounded_{MAX_BARS_SEARCH}_bars'] = (
        fixed_n_extreme(tf['High'], MAX_BARS_SEARCH, 'max'),
        fixed_n_extreme(tf['Low'], MAX_BARS_SEARCH, 'min'),
    )
    return horizons

def add_target_touch_stats(tf):
    """Reach-rate & speed for 'time to Nx the candle's own range' (magnitude is trivial by
    construction here, so this reports probability + bars-to-reach instead)."""
    rng = (tf['High'] - tf['Low']).values
    target_up = tf['Close'].values + TARGET_MULTIPLE * rng
    target_dn = tf['Close'].values - TARGET_MULTIPLE * rng
    bars_up = first_touch_bars(tf['High'].values, target_up, MAX_BARS_SEARCH, 'up')
    bars_dn = first_touch_bars(tf['Low'].values, target_dn, MAX_BARS_SEARCH, 'down')
    return bars_up, bars_dn

## Step 4 — Continuation & retracement math

In [ ]:
def continuation_pct(close, fwd_max_high, fwd_min_low):
    upside_pct = (fwd_max_high - close) / close * 100
    downside_pct = (close - fwd_min_low) / close * 100
    return upside_pct, downside_pct

def level_retracement(close, level, fwd_max_high, fwd_min_low):
    """For each row: does forward price come back to `level`, and how deep (0-100%)?"""
    close = np.asarray(close, dtype=float)
    level = np.asarray(level, dtype=float)
    fwd_max_high = np.asarray(fwd_max_high, dtype=float)
    fwd_min_low = np.asarray(fwd_min_low, dtype=float)

    dist = np.abs(close - level)
    touched = np.zeros(len(close), dtype=bool)
    depth = np.full(len(close), np.nan)
    valid = ~np.isnan(close) & ~np.isnan(level)

    already = valid & (dist == 0)
    touched[already] = True
    depth[already] = 100.0

    below = valid & (level < close) & ~already & ~np.isnan(fwd_min_low)
    achieved = np.clip(close[below] - fwd_min_low[below], 0, dist[below])
    depth[below] = np.where(dist[below] > 0, achieved / dist[below] * 100, 0)
    touched[below] = fwd_min_low[below] <= level[below]

    above = valid & (level > close) & ~already & ~np.isnan(fwd_max_high)
    achieved = np.clip(fwd_max_high[above] - close[above], 0, dist[above])
    depth[above] = np.where(dist[above] > 0, achieved / dist[above] * 100, 0)
    touched[above] = fwd_max_high[above] >= level[above]

    return touched, depth

def near_edge_of_prior_range(close, prior_low, prior_high):
    """Boundary of the prior candle's range you'd need to cross to re-enter it.
    If close is already inside [prior_low, prior_high], distance is 0 (already retraced)."""
    close = np.asarray(close, dtype=float)
    prior_low = np.asarray(prior_low, dtype=float)
    prior_high = np.asarray(prior_high, dtype=float)
    return np.where(close > prior_high, prior_high, np.where(close < prior_low, prior_low, close))

## Step 5 — Full per-timeframe pipeline

In [ ]:
def process_timeframe(df1, rule):
    print(f'--- {rule} ---')
    tf = build_timeframe(df1, rule)
    tf = classify_candles(tf)
    tf = add_open_position(tf)
    print(f'  {len(tf):,} candles built')

    horizons = add_horizon_extremes(tf)
    bars_up, bars_dn = add_target_touch_stats(tf)
    tf[f'bars_to_{TARGET_MULTIPLE}x_range_up'] = bars_up
    tf[f'bars_to_{TARGET_MULTIPLE}x_range_down'] = bars_dn

    cont_rows, retr_rows = [], []
    close = tf['Close']
    for hz_label, (fwd_max_high, fwd_min_low) in horizons.items():
        up_pct, dn_pct = continuation_pct(close, fwd_max_high, fwd_min_low)
        cont_rows.append(pd.DataFrame({
            'timeframe': rule, 'horizon': hz_label, 'candle_type': tf['candle_type'].values,
            'Hour_ET': tf['Hour_ET'].values, 'Session': tf['Session'].values,
            'open_position': tf['open_position'].values, 'direction': tf['direction'].values,
            'upside_pct': up_pct.values, 'downside_pct': dn_pct.values,
        }))
        level_specs = {
            'prior_open': tf['prior_open'].values,
            'prior_mid': tf['prior_mid'].values,
            'prior_range': near_edge_of_prior_range(close.values, tf['prior_low'].values, tf['prior_high'].values),
        }
        for level_name, level_vals in level_specs.items():
            touched, depth = level_retracement(close.values, level_vals, fwd_max_high.values, fwd_min_low.values)
            retr_rows.append(pd.DataFrame({
                'timeframe': rule, 'horizon': hz_label, 'level': level_name, 'candle_type': tf['candle_type'].values,
                'Hour_ET': tf['Hour_ET'].values, 'Session': tf['Session'].values,
                'open_position': tf['open_position'].values, 'direction': tf['direction'].values,
                'touched': touched, 'depth_pct': depth,
            }))

    cont_long = pd.concat(cont_rows, ignore_index=True)
    retr_long = pd.concat(retr_rows, ignore_index=True)

    group_cols = ['timeframe', 'horizon', 'candle_type', 'Hour_ET', 'open_position']
    cont_summary = cont_long.groupby(group_cols, observed=True).agg(
        n=('upside_pct', 'size'),
        upside_median=('upside_pct', 'median'), upside_mean=('upside_pct', 'mean'), upside_max=('upside_pct', 'max'),
        downside_median=('downside_pct', 'median'), downside_mean=('downside_pct', 'mean'), downside_max=('downside_pct', 'max'),
    ).reset_index()
    cont_summary = cont_summary[cont_summary['n'] >= MIN_SAMPLE_SIZE].round(3)

    retr_group_cols = group_cols + ['level']
    retr_summary = retr_long.groupby(retr_group_cols, observed=True).agg(
        n=('touched', 'size'), touch_rate_pct=('touched', 'mean'),
        depth_median=('depth_pct', 'median'), depth_mean=('depth_pct', 'mean'), depth_max=('depth_pct', 'max'),
    ).reset_index()
    retr_summary['touch_rate_pct'] = (retr_summary['touch_rate_pct'] * 100).round(1)
    retr_summary = retr_summary[retr_summary['n'] >= MIN_SAMPLE_SIZE].round(3)

    target_df = pd.DataFrame({
        'candle_type': tf['candle_type'].values, 'Hour_ET': tf['Hour_ET'].values,
        'open_position': tf['open_position'].values, 'direction': tf['direction'].values,
        'bars_up': bars_up, 'bars_dn': bars_dn,
    })
    target_summary = target_df.groupby(['candle_type', 'Hour_ET', 'open_position'], observed=True).agg(
        n=('bars_up', 'size'),
        reach_rate_up_pct=('bars_up', lambda s: (s > 0).mean() * 100),
        bars_up_median=('bars_up', lambda s: s[s > 0].median()),
        bars_up_mean=('bars_up', lambda s: s[s > 0].mean()),
        reach_rate_dn_pct=('bars_dn', lambda s: (s > 0).mean() * 100),
        bars_dn_median=('bars_dn', lambda s: s[s > 0].median()),
        bars_dn_mean=('bars_dn', lambda s: s[s > 0].mean()),
    ).reset_index()
    target_summary.insert(0, 'timeframe', rule)
    target_summary.insert(1, 'target_multiple', TARGET_MULTIPLE)
    target_summary = target_summary[target_summary['n'] >= MIN_SAMPLE_SIZE].round(3)

    del cont_long, retr_long
    gc.collect()
    return tf, cont_summary, retr_summary, target_summary

## Run — load data & build results for every timeframe

This is the slow cell (does the full classification + horizon math for 5m/15m/30m/1h). Everything below just reads from the `results` dict it builds.

In [ ]:
print('Loading data...')
df = pd.read_csv(FILE_PATH)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
df = df.loc[START_DATE:END_DATE]
print(f'Loaded {len(df):,} 1-min rows | {df.index[0]} -> {df.index[-1]}')

results = {}
for rule in TIMEFRAMES:
    tf, cont, retr, target = process_timeframe(df, rule)
    results[rule] = {'events': tf, 'continuation': cont, 'retracement': retr, 'target_touch': target}

print('\nBuilt results for:', list(results.keys()))

## Results — candle type distribution

Worth checking the "Unclassified" % per timeframe — if it's high, loosen the thresholds in the config cell and re-run.

In [ ]:
for rule in TIMEFRAMES:
    counts = results[rule]['events']['candle_type'].value_counts()
    pct_unclassified = counts.get('Unclassified', 0) / counts.sum() * 100
    print(f'\n{rule}  (n={counts.sum():,}, Unclassified={pct_unclassified:.1f}%)')
    print(counts.to_string())

## Results — Continuation explorer

Edit `SELECT_TIMEFRAME` / `SELECT_HORIZON` / `SELECT_OPEN_POS` below and re-run this + the next cell to look at a different slice. Table is sorted by the highest median move (in the candle's own/aligned direction).

In [ ]:
SELECT_TIMEFRAME = '15min'          # '5min' | '15min' | '30min' | '1h'
SELECT_HORIZON    = 'fixed_20_bars'  # 'rest_of_session' | 'fixed_5_bars' | 'fixed_20_bars' | f'bounded_{MAX_BARS_SEARCH}_bars'
SELECT_OPEN_POS   = 'Lower third'    # one of SEG_LABELS

def aligned_pct_row(row):
    """Continuation in the candle's own/expected direction: bull types -> upside,
    bear types -> downside, neutral types (doji/spinning top) -> the larger of the two."""
    bias = CANDLE_BIAS.get(row['candle_type'], 'neutral')
    if bias == 'bull':
        return row['upside_median']
    if bias == 'bear':
        return row['downside_median']
    return max(row['upside_median'], row['downside_median'])

cont = results[SELECT_TIMEFRAME]['continuation']
sub = cont[(cont['horizon'] == SELECT_HORIZON) & (cont['open_position'] == SELECT_OPEN_POS)].copy()
sub['aligned_median_pct'] = sub.apply(aligned_pct_row, axis=1)
sub.sort_values('aligned_median_pct', ascending=False).head(20)

In [ ]:
pivot = sub.pivot_table(index='candle_type', columns='Hour_ET', values='aligned_median_pct', aggfunc='mean')

fig, ax = plt.subplots(figsize=(15, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=0)
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index, fontsize=8)
ax.set_title(f'Median aligned continuation % | {SELECT_TIMEFRAME} | {SELECT_HORIZON} | open in {SELECT_OPEN_POS}')
plt.colorbar(im, ax=ax, label='%')
plt.tight_layout()
plt.show()

## Results — Retracement explorer

Touch-rate = % of candles where price came back to the level within the chosen horizon. Depth = how much of the distance back to that level got closed, on average (100% = fully retraced or beyond).

In [ ]:
SELECT_LEVEL = 'prior_open'   # 'prior_open' | 'prior_mid' | 'prior_range'

retr = results[SELECT_TIMEFRAME]['retracement']
sub_r = retr[(retr['horizon'] == SELECT_HORIZON) & (retr['open_position'] == SELECT_OPEN_POS) & (retr['level'] == SELECT_LEVEL)]
sub_r.sort_values('touch_rate_pct', ascending=False).head(20)

In [ ]:
pivot_r = sub_r.pivot_table(index='candle_type', columns='Hour_ET', values='touch_rate_pct', aggfunc='mean')

fig, ax = plt.subplots(figsize=(15, 6))
im = ax.imshow(pivot_r.values, aspect='auto', cmap='Blues', vmin=0, vmax=100)
ax.set_xticks(range(len(pivot_r.columns))); ax.set_xticklabels(pivot_r.columns)
ax.set_yticks(range(len(pivot_r.index))); ax.set_yticklabels(pivot_r.index, fontsize=8)
ax.set_title(f'Retracement touch-rate % to {SELECT_LEVEL} | {SELECT_TIMEFRAME} | {SELECT_HORIZON} | open in {SELECT_OPEN_POS}')
plt.colorbar(im, ax=ax, label='touch rate %')
plt.tight_layout()
plt.show()

## Results — Target-touch explorer (time to 2x the candle's own range)

This is reach-rate & speed, not magnitude (see note in Step 3) — by definition, price sits *at* the target once it hits. If you change `TARGET_MULTIPLE` in the config cell, this is whatever multiple you set.

In [ ]:
tt = results[SELECT_TIMEFRAME]['target_touch']
sub_t = tt[tt['open_position'] == SELECT_OPEN_POS].sort_values('reach_rate_up_pct', ascending=False)
sub_t.head(20)

## Save results to CSV (optional)

Run this if you want the summaries/events on disk for Excel or further slicing outside this notebook.

In [ ]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

for rule in TIMEFRAMES:
    r = results[rule]
    r['continuation'].to_csv(f'{OUT_DIR}/continuation_summary_{rule}.csv', index=False)
    r['retracement'].to_csv(f'{OUT_DIR}/retracement_summary_{rule}.csv', index=False)
    r['target_touch'].to_csv(f'{OUT_DIR}/target_touch_summary_{rule}.csv', index=False)
    if SAVE_EVENT_LEVEL:
        r['events'].to_csv(f'{OUT_DIR}/candle_events_{rule}.csv')
    print(f'Saved {rule} summaries' + (' + events' if SAVE_EVENT_LEVEL else ''))

print('\nDone.')